# Classifier Kecocokan CV ↔ Lowongan — DirekrutAI

Latih + **banding 2 base model**, lalu export ke **ONNX** buat diserve murah di Heroku (CPU).

> **Tugasnya apa?** Model ini menilai **pasangan (CV, Deskripsi Lowongan) → keputusan**
> (mis. lolos / tolak). Ini yang disebut *cross-encoder* — dan ini persis tugas
> *fit scoring* yang sekarang dikerjain LLM eksternal tiap screening.
> Notebook otomatis deteksi: kalau dataset punya kolom deskripsi lowongan, dia
> latih mode **pair**; kalau cuma teks + kategori, dia latih mode **single**.

**Alur notebook ini:**
1. Banding `IndoBERT` vs `multilingual-MiniLM` di dataset yang sama.
2. Ukur akurasi + macro-F1 di test-set held-out → ini yang jawab *"apakah >95% kesampaian"*.
3. Ambil pemenang, export ke ONNX + tokenizer + label map.
4. (Opsional) upload artifact ke object storage biar `ai-engine` bisa narik & serve.

---

### ⚠️ Baca dulu sebelum jalan

**Runtime:** `Runtime > Change runtime type > T4 GPU` (GPU gratis udah cukup).

**Soal bahasa & target 95%:** dataset default (`AzharAli05`) **mayoritas Bahasa Inggris**.
Kalau CV produksi kamu Bahasa Indonesia, akurasi di data nyata bakal **lebih rendah**
dari angka test-set di notebook ini. `multilingual-MiniLM` biasanya transfer lintas-bahasa
lebih baik daripada IndoBERT-yang-dilatih-data-Inggris — makanya dua-duanya dibanding.
Kalau target 95% di CV Indonesia serius, siapin data CV Indonesia berlabel dan gabung ke dataset.

**Kenapa export ONNX?** Serving pakai `onnxruntime` (~16MB, CPU-cepat) bukan `torch` (~2GB),
biar image Docker `ai-engine` tetap ramping dan gak maksa naik tier dyno Heroku.

## 1. Install dependensi

In [ ]:
!pip install -q "transformers==4.44.2" "datasets==3.0.1" "evaluate==0.4.3" \
    "scikit-learn==1.5.2" "optimum[onnxruntime]==1.23.1" "accelerate==0.34.2" "boto3==1.35.61"

## 2. Konfigurasi

Ubah di sini kalau mau ganti dataset / base model / hyperparameter.
Kolom S3 boleh dikosongin — nanti artifact-nya di-zip buat download manual.

In [ ]:
import json
import os
import shutil
from pathlib import Path
import numpy as np

# Dataset tabular berlabel kategori. Gabung sama data CV Indonesia kamu sendiri
# kalau ada -- makin variatif makin bagus (dan makin realistis buat produksi).
DATASET_ID = "AzharAli05/Resume-Screening-Dataset"

# Dua base model yang dibanding. Keduanya encoder kecil -> muat CPU pas di-serve.
BASE_MODELS = {
    "indobert": "indobenchmark/indobert-base-p1",
    "multilingual-minilm": "microsoft/Multilingual-MiniLM-L12-H384",
}

MAX_LEN = 256          # harus SAMA dengan CV_CLASSIFIER_MAX_LEN di ai-engine
EPOCHS = 3
BATCH_SIZE = 16
OUTPUT_DIR = Path("cv_classifier_out")

# Upload artifact ke object storage? Isi kalau iya.
# Ambil nilainya dari: heroku config -a direkrut-ai-ai-engine
S3_ENDPOINT   = os.environ.get("OBJECT_STORAGE_ENDPOINT", "")
S3_ACCESS_KEY = os.environ.get("OBJECT_STORAGE_ACCESS_KEY", "")
S3_SECRET_KEY = os.environ.get("OBJECT_STORAGE_SECRET_KEY", "")
S3_BUCKET     = os.environ.get("OBJECT_STORAGE_BUCKET", "")
S3_PREFIX     = os.environ.get("CV_CLASSIFIER_PREFIX", "models/cv-classifier-v1/")

print("Konfig siap. Dataset:", DATASET_ID)

## 3. Load & siapkan dataset

Kolom teks/label dideteksi otomatis (nama kolom beda-beda antar dataset).
Kalau salah tebak, set manual `text_col` / `label_col` di sel ini.

In [ ]:
from collections import Counter

from datasets import load_dataset

ds = load_dataset(DATASET_ID, split="train")
cols = ds.column_names
print("Kolom dataset:", cols)

def _pick(cands):
    return next((c for c in cols if c.lower() in cands), None)

# Teks utama (CV), pasangan opsional (deskripsi lowongan), dan label.
text_col  = _pick({"resume", "resume_str", "text", "cv"})
pair_col  = _pick({"job_description", "jobdescription", "jd", "job_desc"})
label_col = _pick({"decision", "category", "label", "job_category", "class", "fit"})

# Override manual di sini kalau deteksi otomatis salah:
# text_col, pair_col, label_col = "Resume", "Job_Description", "Decision"

if text_col is None or label_col is None:
    raise ValueError(f"Gagal deteksi kolom dari {cols}. Set manual di sel ini.")

MODE = "pair" if pair_col else "single"
print(f"MODE={MODE}  text_col={text_col}  pair_col={pair_col}  label_col={label_col}")

keep = [c for c in (text_col, pair_col, label_col) if c]
ds = ds.filter(lambda r: all(r[c] is not None and str(r[c]).strip() for c in keep))

labels = sorted(set(str(x) for x in ds[label_col]))
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

dist = Counter(str(x) for x in ds[label_col])
print(f"\n{len(labels)} kelas -> {labels}")
print("Distribusi label:", dict(dist))
if len(labels) < 2:
    raise ValueError("Cuma 1 kelas -- gak bisa dilatih. Cek kolom label.")

def _encode(row):
    out = {"labels": label2id[str(row[label_col])], "text": str(row[text_col])}
    if pair_col:
        out["text_pair"] = str(row[pair_col])
    return out

ds = ds.map(_encode, remove_columns=cols)
split = ds.train_test_split(test_size=0.2, seed=42, stratify_by_column="labels")
train_ds, test_ds = split["train"], split["test"]
print(f"\ntrain={len(train_ds)}  test={len(test_ds)}")

## 4. Fungsi training

Satu fungsi dipakai buat kedua base model, biar perbandingannya apple-to-apple
(hyperparameter, split, dan metrik persis sama).

In [ ]:
import evaluate
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

def train_one(name, base_id):
    print(f"\n{'='*60}\nLATIH: {name} ({base_id})\n{'='*60}")
    tokenizer = AutoTokenizer.from_pretrained(base_id)

    def tok(batch):
        # Mode pair -> cross-encoder: CV dan deskripsi lowongan masuk BARENGAN
        # sebagai satu pasangan, dipisah [SEP]. Ini yang bikin model belajar
        # KECOCOKAN, bukan cuma "resume ini bagus/nggak".
        if MODE == "pair":
            return tokenizer(batch["text"], batch["text_pair"], truncation=True,
                             max_length=MAX_LEN, padding="max_length")
        return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN, padding="max_length")

    train_tok = train_ds.map(tok, batched=True)
    test_tok = test_ds.map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        base_id, num_labels=len(id2label), id2label=id2label, label2id=label2id)

    acc, f1 = evaluate.load("accuracy"), evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, y = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {"accuracy": acc.compute(predictions=pred, references=y)["accuracy"],
                "macro_f1": f1.compute(predictions=pred, references=y, average="macro")["f1"]}

    out = OUTPUT_DIR / name
    args = TrainingArguments(
        output_dir=str(out), num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        eval_strategy="epoch", save_strategy="no", learning_rate=2e-5,
        weight_decay=0.01, logging_steps=50, report_to="none")

    trainer = Trainer(model=model, args=args, train_dataset=train_tok,
                      eval_dataset=test_tok, compute_metrics=compute_metrics)
    trainer.train()
    m = trainer.evaluate()
    print(f"HASIL {name}: acc={m['eval_accuracy']:.4f}  macro_f1={m['eval_macro_f1']:.4f}")

    hf_dir = out / "hf"
    trainer.save_model(str(hf_dir))
    tokenizer.save_pretrained(str(hf_dir))
    return {"name": name, "hf_dir": hf_dir,
            "accuracy": m["eval_accuracy"], "macro_f1": m["eval_macro_f1"]}

## 5. Jalankan perbandingan  ← **ini inti langkah 1**

Sel ini yang paling lama (latih 2 model). Hasilnya: tabel perbandingan +
pemenang, plus peringatan kalau belum tembus 95%.

In [ ]:
results = [train_one(n, b) for n, b in BASE_MODELS.items()]

print(f"\n{'='*60}\nPERBANDINGAN\n{'='*60}")
for r in sorted(results, key=lambda x: -x["macro_f1"]):
    print(f"  {r['name']:22s} acc={r['accuracy']:.4f}  macro_f1={r['macro_f1']:.4f}")

winner = max(results, key=lambda x: x["macro_f1"])
print(f"\nPEMENANG: {winner['name']} (macro_f1={winner['macro_f1']:.4f})")
if winner["macro_f1"] < 0.95:
    print("  ! Belum >95%. Opsi: tambah data (khususnya CV Indonesia),")
    print("    naikkan EPOCHS, atau coba base model lain.")
else:
    print("  OK, tembus target 95%.")

## 6. Export pemenang → ONNX

Yang dipakai runtime cuma 3 file: `model.onnx`, `tokenizer.json`, `labels.json`.

In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification

onnx_dir = OUTPUT_DIR / "onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)

ort = ORTModelForSequenceClassification.from_pretrained(str(winner["hf_dir"]), export=True)
ort.save_pretrained(str(onnx_dir))
AutoTokenizer.from_pretrained(str(winner["hf_dir"])).save_pretrained(str(onnx_dir))
(onnx_dir / "labels.json").write_text(
    json.dumps({str(k): v for k, v in id2label.items()}, ensure_ascii=False))

# meta.json dibaca ai-engine biar tau model ini butuh pasangan (CV + lowongan)
# atau cukup satu teks.
(onnx_dir / "meta.json").write_text(
    json.dumps({"mode": MODE, "max_len": MAX_LEN}, ensure_ascii=False))

print("Isi folder export:", sorted(os.listdir(onnx_dir)))

## 7. Upload artifact

Kalau env S3 diisi → langsung upload. Kalau kosong → di-zip buat download manual
(lalu upload sendiri ke bucket).

In [ ]:
if S3_ENDPOINT and S3_BUCKET:
    import boto3
    from botocore.config import Config
    s3 = boto3.client("s3", endpoint_url=S3_ENDPOINT,
                      aws_access_key_id=S3_ACCESS_KEY, aws_secret_access_key=S3_SECRET_KEY,
                      region_name="us-east-1", config=Config(signature_version="s3v4"))
    for fname in ("model.onnx", "tokenizer.json", "labels.json", "meta.json"):
        src = onnx_dir / fname
        if src.exists():
            s3.upload_file(str(src), S3_BUCKET, S3_PREFIX + fname)
            print("Uploaded", S3_PREFIX + fname)
    print(f"\nSelesai. Jalankan:\n  heroku config:set CV_CLASSIFIER_PREFIX={S3_PREFIX} -a direkrut-ai-ai-engine")
else:
    shutil.make_archive("cv_classifier_onnx", "zip", onnx_dir)
    print("Env S3 kosong -> artifact di-zip: cv_classifier_onnx.zip")
    print("Download dari panel Files Colab, lalu upload 3 file itu ke bucket kamu.")

## 8. Langkah setelah notebook ini

1. Pastikan 4 file (`model.onnx`, `tokenizer.json`, `labels.json`, `meta.json`) ada di bucket
   dengan prefix yang sama, mis. `models/cv-classifier-v1/`.
2. Set env di Heroku:
   ```
   heroku config:set CV_CLASSIFIER_PREFIX=models/cv-classifier-v1/ -a direkrut-ai-ai-engine
   ```
3. Redeploy `ai-engine`. Endpoint `POST /v1/cv-classifier/classify` langsung hidup
   (sebelum model ada, endpoint balikin 503 yang jelas — aman).
4. Uji:
   ```bash
   curl -X POST https://<ai-engine>/v1/cv-classifier/classify \
     -H "X-Internal-Api-Key: $KEY" -H "Content-Type: application/json" \
     -d '{"text":"Software Engineer 5 tahun pengalaman Go dan PostgreSQL",
          "job_description":"Backend Engineer, butuh Go dan PostgreSQL"}'
   ```
   (`job_description` wajib kalau model dilatih mode **pair**.)

**Retrain berikutnya:** jangan latih per-input user (risiko *catastrophic forgetting*
dan *data poisoning*). Kumpulin data berlabel, retrain batch berkala di notebook ini,
bandingkan macro-F1 dengan model lama, dan **promote hanya kalau menang**.
Naikkan versi prefix (`cv-classifier-v2/`) biar gampang rollback.